In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q faiss-cpu


In [ ]:
import re
import string

from glob import glob
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import torch
import faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
import os
import wandb

wandb_mode = "online" if os.environ.get("WANDB_API_KEY") else "disabled"

run = wandb.init(
    project="23f2000775-t22026",
    name="milestone3-rag",
    mode=wandb_mode,
    config={
        "embedding_model": embedding_model_name,
        "llm": llm_name,
        "top_k": top_k,
        "sample_size": sample_size
    },
    reinit=True
)

wandb.log({
    "MAP@3": rag_map3,
    "Accuracy": rag_accuracy
})

run.finish()

In [ ]:
train_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

train = pd.read_csv(train_path)

print(train.shape)
train.head()

In [ ]:
train.isnull().sum()

In [ ]:
text_columns = ['prompt', 'A', 'B', 'C', 'D', 'E']

train[text_columns] = train[text_columns].fillna('')

In [ ]:
def clean_text(text):
    text = text.lower()

    for mark in string.punctuation:
        text = text.replace(mark, ' ')

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
for column in text_columns:
    train[column + '_clean'] = train[column].apply(clean_text)

train[['prompt', 'prompt_clean']].head()

In [ ]:
train['prompt_tokens'] = train['prompt_clean'].str.split()

train[['prompt_clean', 'prompt_tokens']].head()

In [ ]:
def average_precision_at_3(actual, predicted):
    predicted = predicted[:3]

    if actual not in predicted:
        return 0

    rank = predicted.index(actual) + 1

    return 1 / rank

In [ ]:
def map_at_3(actual_answers, predictions):
    scores = []

    for actual, predicted in zip(actual_answers, predictions):
        score = average_precision_at_3(actual, predicted)
        scores.append(score)

    return np.mean(scores)

In [ ]:
clean_columns = [
    'prompt_clean',
    'A_clean',
    'B_clean',
    'C_clean',
    'D_clean',
    'E_clean'
]

all_text = []

for column in clean_columns:
    all_text.extend(train[column].tolist())

print('Number of text samples:', len(all_text))

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf.fit(all_text)

print('TF-IDF vocabulary size:', len(tfidf.vocabulary_))

In [ ]:
option_labels = ['A', 'B', 'C', 'D', 'E']
tfidf_predictions = []
tfidf_scores = []

for _, row in train.iterrows():
    prompt_vector = tfidf.transform([row['prompt_clean']])

    options = [
        row['A_clean'],
        row['B_clean'],
        row['C_clean'],
        row['D_clean'],
        row['E_clean']
    ]

    option_vectors = tfidf.transform(options)

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    tfidf_predictions.append(prediction)
    tfidf_scores.append(similarities)

In [ ]:
train['tfidf_prediction'] = tfidf_predictions

tfidf_map = map_at_3(
    train['answer'].tolist(),
    tfidf_predictions
)

print('TF-IDF MAP@3:', round(tfidf_map, 4))

train[
    ['id', 'answer', 'tfidf_prediction']
].head(10)

In [ ]:
sentences = []

for column in clean_columns:
    for text in train[column]:
        sentences.append(text.split())

print('Number of sentences:', len(sentences))

In [ ]:
word2vec = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

print('Word2Vec vocabulary size:', len(word2vec.wv))

In [ ]:
def get_sentence_vector(text):
    vectors = []

    for word in text.split():
        if word in word2vec.wv:
            vectors.append(word2vec.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [ ]:
word2vec_predictions = []
word2vec_scores = []

for _, row in train.iterrows():
    prompt_vector = get_sentence_vector(
        row['prompt_clean']
    ).reshape(1, -1)

    option_vectors = []

    for label in option_labels:
        option_vector = get_sentence_vector(
            row[label + '_clean']
        )
        option_vectors.append(option_vector)

    option_vectors = np.array(option_vectors)

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    word2vec_predictions.append(prediction)
    word2vec_scores.append(similarities)

In [ ]:
train['word2vec_prediction'] = word2vec_predictions

word2vec_map = map_at_3(
    train['answer'].tolist(),
    word2vec_predictions
)

print('Word2Vec MAP@3:', round(word2vec_map, 4))

## milestone4

In [1]:
import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deberta_name = "microsoft/deberta-v3-small"
roberta_name = "roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_name)
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_name,
    num_labels=5
).to(device)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_name,
    num_labels=5
).to(device)

deberta_model.eval()
roberta_model.eval()

label_map = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

print("Setup completed.")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Setup completed.


In [2]:
row = train.loc[25]

text = (
    f"{row['prompt']} [SEP] "
    f"A: {row['A']} "
    f"B: {row['B']} "
    f"C: {row['C']} "
    f"D: {row['D']} "
    f"E: {row['E']}"
)

inputs = deberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = deberta_model(**inputs)

probs = torch.softmax(outputs.logits, dim=-1)[0].cpu()

labels = ["A", "B", "C", "D", "E"]

best_idx = torch.argmax(probs).item()

print("Answer:", labels[best_idx])
print("Probability:", round(probs[best_idx].item(), 4))

Answer: D
Probability: 0.2781


In [3]:
row = train.loc[25]

text = (
    f"{row['prompt']} [SEP] "
    f"A: {row['A']} "
    f"B: {row['B']} "
    f"C: {row['C']} "
    f"D: {row['D']} "
    f"E: {row['E']}"
)

inputs = deberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = deberta_model(**inputs)

probs = torch.softmax(outputs.logits, dim=-1)[0]

labels = ["A", "B", "C", "D", "E"]

best_idx = torch.argmax(probs).item()

print(f"{labels[best_idx]}, {probs[best_idx].item():.4f}")

D, 0.2781


In [4]:
row = train.loc[25]

text = (
    f"{row['prompt']} [SEP] "
    f"A: {row['A']} "
    f"B: {row['B']} "
    f"C: {row['C']} "
    f"D: {row['D']} "
    f"E: {row['E']}"
)

deberta_inputs = deberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

deberta_inputs = {k: v.to(device) for k, v in deberta_inputs.items()}

roberta_inputs = roberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

roberta_inputs = {k: v.to(device) for k, v in roberta_inputs.items()}

with torch.no_grad():
    deberta_probs = torch.softmax(
        deberta_model(**deberta_inputs).logits,
        dim=-1
    )[0]

    roberta_probs = torch.softmax(
        roberta_model(**roberta_inputs).logits,
        dim=-1
    )[0]

avg_probs = (deberta_probs + roberta_probs) / 2

labels = ["A", "B", "C", "D", "E"]

best_idx = torch.argmax(avg_probs).item()

print(f"{labels[best_idx]}, {avg_probs[best_idx].item():.4f}")

D, 0.2467


In [5]:
weighted_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

labels = ["A", "B", "C", "D", "E"]

best_idx = torch.argmax(weighted_probs).item()

print(f"{labels[best_idx]}, {weighted_probs[best_idx].item():.4f}")


D, 0.2593


In [6]:
labels = ["A", "B", "C", "D", "E"]

top3_idx = torch.argsort(weighted_probs, descending=True)[:3]

top3 = [labels[i] for i in top3_idx]

print(" ".join(top3))

D A B


In [7]:
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(len(test))

500


In [8]:
labels = ["A", "B", "C", "D", "E"]

changed = 0

for i in range(50):

    row = test.loc[i]

    original = (
        f"{row['prompt']} [SEP] "
        f"A: {row['A']} "
        f"B: {row['B']} "
        f"C: {row['C']} "
        f"D: {row['D']} "
        f"E: {row['E']}"
    )

    augmented = (
        "Answer the following multiple-choice question carefully: "
        + original
    )

    original_inputs = deberta_tokenizer(
        original,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    original_inputs = {
        k: v.to(device)
        for k, v in original_inputs.items()
    }

    augmented_inputs = deberta_tokenizer(
        augmented,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    augmented_inputs = {
        k: v.to(device)
        for k, v in augmented_inputs.items()
    }

    with torch.no_grad():

        original_probs = torch.softmax(
            deberta_model(**original_inputs).logits,
            dim=-1
        )[0]

        augmented_probs = torch.softmax(
            deberta_model(**augmented_inputs).logits,
            dim=-1
        )[0]

    original_top1 = torch.argmax(original_probs).item()

    tta_probs = (original_probs + augmented_probs) / 2

    tta_top1 = torch.argmax(tta_probs).item()

    if original_top1 != tta_top1:
        changed += 1

print(changed)

6


In [10]:
different = 0

for i in range(100):

    row = test.loc[i]

    text = (
        f"{row['prompt']} [SEP] "
        f"A: {row['A']} "
        f"B: {row['B']} "
        f"C: {row['C']} "
        f"D: {row['D']} "
        f"E: {row['E']}"
    )

    deberta_inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    deberta_inputs = {k: v.to(device) for k, v in deberta_inputs.items()}

    roberta_inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    roberta_inputs = {k: v.to(device) for k, v in roberta_inputs.items()}

    with torch.no_grad():
        deberta_probs = torch.softmax(
            deberta_model(**deberta_inputs).logits,
            dim=-1
        )[0]

        roberta_probs = torch.softmax(
            roberta_model(**roberta_inputs).logits,
            dim=-1
        )[0]

    deberta_top1 = torch.argmax(deberta_probs).item()

    ensemble_probs = 0.7 * deberta_probs + 0.3 * roberta_probs
    ensemble_top1 = torch.argmax(ensemble_probs).item()

    if deberta_top1 != ensemble_top1:
        different += 1

print("Rows with different Top-1 predictions:", different)

Rows with different Top-1 predictions: 8


In [9]:
positive_gain = 0

for i in range(100):

    row = test.loc[i]

    text = (
        f"{row['prompt']} [SEP] "
        f"A: {row['A']} "
        f"B: {row['B']} "
        f"C: {row['C']} "
        f"D: {row['D']} "
        f"E: {row['E']}"
    )

    deberta_inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    deberta_inputs = {k: v.to(device) for k, v in deberta_inputs.items()}

    roberta_inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    roberta_inputs = {k: v.to(device) for k, v in roberta_inputs.items()}

    with torch.no_grad():

        deberta_probs = torch.softmax(
            deberta_model(**deberta_inputs).logits,
            dim=-1
        )[0]

        roberta_probs = torch.softmax(
            roberta_model(**roberta_inputs).logits,
            dim=-1
        )[0]

    ensemble_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    deberta_confidence = torch.max(deberta_probs).item()
    ensemble_confidence = torch.max(ensemble_probs).item()

    if ensemble_confidence > deberta_confidence:
        positive_gain += 1

print(positive_gain)

0


In [11]:
positive_gain = 0

for i in range(100):

    row = test.loc[i]

    text = (
        f"{row['prompt']} [SEP] "
        f"A: {row['A']} "
        f"B: {row['B']} "
        f"C: {row['C']} "
        f"D: {row['D']} "
        f"E: {row['E']}"
    )

    deberta_inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    deberta_inputs = {k: v.to(device) for k, v in deberta_inputs.items()}

    roberta_inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    roberta_inputs = {k: v.to(device) for k, v in roberta_inputs.items()}

    with torch.no_grad():

        deberta_probs = torch.softmax(
            deberta_model(**deberta_inputs).logits,
            dim=-1
        )[0]

        roberta_probs = torch.softmax(
            roberta_model(**roberta_inputs).logits,
            dim=-1
        )[0]

    ensemble_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    deberta_confidence = torch.max(deberta_probs).item()
    ensemble_confidence = torch.max(ensemble_probs).item()

    if ensemble_confidence > deberta_confidence:
        positive_gain += 1

print(positive_gain)

0


In [12]:
different = 0

for i in range(100):

    row = test.loc[i]

    text = (
        f"{row['prompt']} [SEP] "
        f"A: {row['A']} "
        f"B: {row['B']} "
        f"C: {row['C']} "
        f"D: {row['D']} "
        f"E: {row['E']}"
    )

    deberta_inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    deberta_inputs = {k: v.to(device) for k, v in deberta_inputs.items()}

    roberta_inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    roberta_inputs = {k: v.to(device) for k, v in roberta_inputs.items()}

    with torch.no_grad():
        deberta_probs = torch.softmax(
            deberta_model(**deberta_inputs).logits,
            dim=-1
        )[0]

        roberta_probs = torch.softmax(
            roberta_model(**roberta_inputs).logits,
            dim=-1
        )[0]

    deberta_top1 = torch.argmax(deberta_probs).item()

    ensemble_probs = 0.7 * deberta_probs + 0.3 * roberta_probs
    ensemble_top1 = torch.argmax(ensemble_probs).item()

    if deberta_top1 != ensemble_top1:
        different += 1

print("Rows with different Top-1 predictions:", different)

Rows with different Top-1 predictions: 8


In [13]:
different = 0

for i in range(100):

    row = test.loc[i]

    text = (
        f"{row['prompt']} [SEP] "
        f"A: {row['A']} "
        f"B: {row['B']} "
        f"C: {row['C']} "
        f"D: {row['D']} "
        f"E: {row['E']}"
    )

    deberta_inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    deberta_inputs = {k: v.to(device) for k, v in deberta_inputs.items()}

    roberta_inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    roberta_inputs = {k: v.to(device) for k, v in roberta_inputs.items()}

    with torch.no_grad():
        deberta_probs = torch.softmax(
            deberta_model(**deberta_inputs).logits,
            dim=-1
        )[0]

        roberta_probs = torch.softmax(
            roberta_model(**roberta_inputs).logits,
            dim=-1
        )[0]

    deberta_top1 = torch.argmax(deberta_probs).item()

    ensemble_probs = 0.7 * deberta_probs + 0.3 * roberta_probs
    ensemble_top1 = torch.argmax(ensemble_probs).item()

    if deberta_top1 != ensemble_top1:
        different += 1

print("Rows with different Top-1 predictions:", different)

Rows with different Top-1 predictions: 8


In [14]:
labels = ["A", "B", "C", "D", "E"]

changed = 0

for i in range(100):

    row = test.loc[i]

    text = (
        f"{row['prompt']} [SEP] "
        f"A: {row['A']} "
        f"B: {row['B']} "
        f"C: {row['C']} "
        f"D: {row['D']} "
        f"E: {row['E']}"
    )

    deberta_inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    deberta_inputs = {k: v.to(device) for k, v in deberta_inputs.items()}

    roberta_inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    roberta_inputs = {k: v.to(device) for k, v in roberta_inputs.items()}

    with torch.no_grad():

        deberta_probs = torch.softmax(
            deberta_model(**deberta_inputs).logits,
            dim=-1
        )[0]

        roberta_probs = torch.softmax(
            roberta_model(**roberta_inputs).logits,
            dim=-1
        )[0]

    ensemble_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    deberta_top3 = torch.argsort(deberta_probs, descending=True)[:3].tolist()
    ensemble_top3 = torch.argsort(ensemble_probs, descending=True)[:3].tolist()

    if deberta_top3 != ensemble_top3:
        changed += 1

print(changed)

29


In [15]:
labels = ["A", "B", "C", "D", "E"]

def average_precision_at_3(actual, predicted):
    if actual in predicted:
        return 1 / (predicted.index(actual) + 1)
    return 0

scores = []

for i in range(100):

    row = train.loc[i]

    text = (
        f"{row['prompt']} [SEP] "
        f"A: {row['A']} "
        f"B: {row['B']} "
        f"C: {row['C']} "
        f"D: {row['D']} "
        f"E: {row['E']}"
    )

    deberta_inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    deberta_inputs = {k: v.to(device) for k, v in deberta_inputs.items()}

    roberta_inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    roberta_inputs = {k: v.to(device) for k, v in roberta_inputs.items()}

    with torch.no_grad():

        deberta_probs = torch.softmax(
            deberta_model(**deberta_inputs).logits,
            dim=-1
        )[0]

        roberta_probs = torch.softmax(
            roberta_model(**roberta_inputs).logits,
            dim=-1
        )[0]

    ensemble_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    top3 = torch.argsort(
        ensemble_probs,
        descending=True
    )[:3].tolist()

    prediction = [labels[j] for j in top3]

    scores.append(
        average_precision_at_3(
            row["answer"],
            prediction
        )
    )

map3 = sum(scores) / len(scores)

print(round(map3, 4))

0.34


In [16]:
weighted_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

labels = ["A", "B", "C", "D", "E"]

best_idx = torch.argmax(weighted_probs).item()

print(f"{labels[best_idx]}, {weighted_probs[best_idx].item():.4f}")

D, 0.2299
